## Installation

In [1]:
#!brew install poppler

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
os.environ["ANTHROPIC_API_KEY"] =""

## Test Extractors

### Rubric Extractor

In [4]:
import fitz  # PyMuPDF

def extract_rubric(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text.strip()

In [5]:
rubric = extract_rubric("../data/ExamQuestionAndRubric.pdf")

In [35]:
rubric

'Exam question \nCase study (3 points): In this exercise, your task is to apply the Bad Actors Modeling strategy to the \nSmartCity solution described in the scenario below: \n• \nChoose 3 different categories among the five in the Bad Actors Modeling strategy (money, \npolitics, entertainment, ideas, self-interest).  \n• \nThen for each of the categories you have chosen: identify and describe one harmful action a \nbad actor could take and how it could negatively impact stakeholders (1-2 sentences for \neach). Actions must be different from each other.  \nScenario: The Swiss company SmartCity wants to help local authorities to better manage city \ninfrastructures and resources by deploying a solution based on the Internet of Things (IoT) - a \nnetwork of objects with computing capabilities connected to the Internet through Wi-Fi or 5G. The \nsolution proposed by SmartCity uses a range of connected devices to monitor waste-bins, traffic flow \nas well as weather conditions throughout a

'Exam question \nCase study (3 points): In this exercise, your task is to apply the Bad Actors Modeling strategy to the \nSmartCity solution described in the scenario below: \n• \nChoose 3 different categories among the five in the Bad Actors Modeling strategy (money, \npolitics, entertainment, ideas, self-interest).  \n• \nThen for each of the categories you have chosen: identify and describe one harmful action a \nbad actor could take and how it could negatively impact stakeholders (1-2 sentences for \neach). Actions must be different from each other.  \nScenario: The Swiss company SmartCity wants to help local authorities to better manage city \ninfrastructures and resources by deploying a solution based on the Internet of Things (IoT) - a \nnetwork of objects with computing capabilities connected to the Internet through Wi-Fi or 5G. The \nsolution proposed by SmartCity uses a range of connected devices to monitor waste-bins, traffic flow \nas well as weather conditions throughout a city. Level monitoring sensors on waste-bins allow waste \ncollection services to optimize their routes. Cameras on major roads and intersections are connected \nto machine learning models to analyze real-time traffic data and dynamically adjust green light times. \nUsing real-time weather data, indoor ventilation and temperature are automatically adjusted to keep \nbuilding occupants comfortable while minimizing energy use. Weather data is also used in predictive \nanalytics to identify health risks and aggregated information is communicated to local communities in \na public app, for instance in case of air pollution or heatwave. Staff from the city authorities can \nmonitor and control the system remotely from a centralized dashboard. SmartCity plans to roll out its \nsolution in a number of European countries with large cities, including France, Germany, Italy and \nSpain.  \n \nGrading rubric \n1 point per bad actor action (total = 3 points) \n• \n0.5 the action described corresponds to the category (check the cheatsheet)  \n• \n0.5 the description makes clear how the action is harmful / negatively impacts stakeholders \nand is specific to SmartCity (i.e. adapted to the scenario)   \nPenalizations:  \n• \n-0.5 if two actions are too similar to each other  \n• \n-0.25 if negative impact is not sufficiently described'

### Student Answer Extractor

In [13]:
import pytesseract
from pdf2image import convert_from_path

def extract_student_answers(paths):
    results = []

    for path in paths:
        pages = convert_from_path(path)
        text = ""

        for page in pages:
            text += pytesseract.image_to_string(page)

        results.append(text.strip())

    return results

In [6]:

from openai import OpenAI
import fitz
import base64
import json
import os

client = OpenAI()


def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    images = []

    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img_bytes = pix.tobytes("png")
        b64 = base64.b64encode(img_bytes).decode("utf-8")
        images.append(f"data:image/png;base64,{b64}")

    return images
def extract_student_transcription(pdf_paths, output_file="student_transcriptions.json"):
    results = []

    for path in pdf_paths:
        images = pdf_to_images(path)

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=[{
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
You are a transcription system for handwritten exam answers.

IMPORTANT:
- Do NOT summarize
- Do NOT interpret
- Do NOT improve grammar
- Preserve structure
- Use [unclear] if needed

Return STRICT JSON:

{
  "transcription": "...",
  "uncertain_words": [],
  "confidence": 0-100
}
"""
                    },
                    *[
                        {
                            "type": "input_image",
                            "image_url": img,
                            "detail": "high"
                        }
                        for img in images
                    ]
                ]
            }]
        )

        raw_output = response.output_text

        try:
            parsed = json.loads(raw_output)
        except:
            parsed = {
                "transcription": raw_output,
                "uncertain_words": [],
                "confidence": 0
            }

        results.append({
            "file": path,
            "result": parsed
        })

    # 💾 SAVE TO FILE
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    return results

In [7]:
students = extract_student_transcription([
        "../data/StudentAnswer-Student1.pdf",
        "../data/StudentAnswer-Student2.pdf",
        "../data/StudentAnswer-Student3.pdf"
    ])

In [28]:
students

{'transcription': 'Money: A competitor of the company Smartcity can hack the centralized monitoring system, this would lead to a mess with the traffic and the bins. The company Smartcity would be negatively impacted in their reputation and this would lead to the loss of money by not signing contract with other European countries. The motivation for the competitor is the money they gain some part of the market.\nPolitics: A competitor of the current "maire (chef de la ville)" can disturb the sensors on waste-bins, the harm would be that citizens wouldn\'t be happy anymore with the council of the city because there would be bins everywhere due to the disruption of the sensors. The motivation for the competitor would be to be elected in the next elections.\nIdeas: A group of people pro-privacy can destroy/cover the cameras used to analyze the real-time traffic arguing that it is not in their ideology to be looked after by ML augmented cameras. The stakeholders harmed would be all the user

#### OCR Extraction

In [24]:
students

['Politics Tictillig amen Cam Vee\n| [ws pero fe ihe ud ji bet\nom y . bt a a7\n\nbeg a have ey pad Pecan\nHe Robhas drm tube Hue vwheable\n\nWeld weticle - furor’, Lomeas , Hutter\nwae be soll Huon oe Oh a “a\ntrtibaiebuat Y Some bad deler aol\nHe wore ellecton ewer bY many ula\nde. Pauses for mahe Hw Wwarte~colle ae,\nLrteot Hu lallredors be He Come Lacobialen\n\nSeren [yes',
 '© Cateye “F Z vl ene?\n\naH lef 7 | waa te bo lA i a or even 14 ulingy iC\n\n6S iMce pl mig at 4 Bowe esol l eS ets Se vf\n\nOr it cen eucoy Tehe dre on authori fies te speek\noN lod “| ia me a\n\n‘ Catyer7 . Sel “interest | | .\n\nSome pepe might vse ot for dl puaformedon | sme\nnt can pobtly veh peed, . They could\niw he posped ea the wes cud ollow {he ao les |\nput could hetm Lee fj” Qoeng le 1 t fetsen\n\nAs tet eM\nwhe might hewve Ce oli kt Ly 2 bave Toe street\n\nent eu yt anil Freve le fi" yre Th We\nond Jem rere Wee being adj Ktedh mutters ie My wih\nagen dence on Thee weefler outside o¢ The enviunat\n

### Teaching Material Extractor

In [8]:
import fitz
import base64
import json
import os
from datetime import datetime
from openai import OpenAI

client = OpenAI()


# ----------------------------
# 1. Convert PDF → images
# ----------------------------
def pdf_to_base64_images(pdf_path: str):
    doc = fitz.open(pdf_path)
    images = []

    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        base64_img = base64.b64encode(img_bytes).decode("utf-8")

        images.append({
            "type": "input_image",
            "image_url": f"data:image/png;base64,{base64_img}",
            "detail": "high"
        })

    return images


# ----------------------------
# 2. Extract structured content
# ----------------------------
def extract_teaching_material(pdf_path: str, save_output: bool = True):

    images = pdf_to_base64_images(pdf_path)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": """
You are extracting teaching material from a PDF.

IMPORTANT:
- Do NOT summarize.
- Extract EVERYTHING visible.
- Preserve structure exactly.
- Include diagram content (WENN diagrams, arrows, boxes).
- If text is in a figure, transcribe it.
- If relationships exist, describe them explicitly.

OUTPUT FORMAT (strict JSON):

{
  "key_concepts": [],
  "definitions": [],
  "diagram_explanations": [],
  "relationships": [],
  "raw_notes": []
}

Rules:
- Be exhaustive
- Do not omit small details
- Do not interpret beyond visible information
"""
                },
                *images
            ]
        }]
    )

    extracted_text = response.output_text

    # ----------------------------
    # 3. Save output for inspection
    # ----------------------------
    if save_output:
        os.makedirs("outputs", exist_ok=True)

        filename = f"outputs/teaching_material_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        try:
            parsed = json.loads(extracted_text)
        except Exception:
            parsed = {"raw_output": extracted_text}

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=2, ensure_ascii=False)

        print(f"[Saved] {filename}")

    return extracted_text

In [9]:
teaching = extract_teaching_material("../data/TeachingResource-BAD_ACTORS_STRATEGY.pdf")

[Saved] outputs/teaching_material_20260421_012425.json


In [5]:
teaching

'- **Fundamentals of the strategy**\n\n  - **Goal**:  \n    - Minimize vulnerabilities from the beginning of software design through risk assessment and mitigation.\n\n  - **When to use it**:  \n    - During the software design phase  \n    - Prior to any updates  \n    - During audits and inspections  \n    - As part of incident response  \n    - While improving the software\n\n- **What is a Bad actor?**\n\n  - Definition:  \n    - A "bad actor" or "threat actor" is a person or group that tries to exploit a system to achieve their own personal goals.\n\n  - Key points:  \n    - Multiple types of bad actors exist.  \n    - They are driven by different motivations.  \n    - Understanding and categorizing these motivations is essential for threat modeling.\n\n- **Assessment questions** (to guide risk analysis)\n\n  - Who would want to misuse the software or use it in an unintended way?  \n  - What would be the consequences of such misuse?  \n  - How could the negative consequences be pre

## Test pipeline

In [15]:
import json, hashlib, os

CACHE_FILE = "cache.json"


def cache_key(text):
    return hashlib.md5(text.encode()).hexdigest()


def load_cache():
    if not os.path.exists(CACHE_FILE):
        return {}
    with open(CACHE_FILE) as f:
        return json.load(f)


def save_cache(data):
    with open(CACHE_FILE, "w") as f:
        json.dump(data, f, indent=2)


def cached_call(key, fn):
    cache = load_cache()

    if key in cache:
        return cache[key]

    result = fn()
    cache[key] = result
    save_cache(cache)

    return result

In [16]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

In [17]:

import logging
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY)
logger = logging.getLogger(__name__)

In [22]:
import time


def normalize(text):
    return " ".join(text.split())


def call_llm(prompt, model, temperature=0.0, retries=3):

    key = cache_key(f"{model}:{temperature}:{normalize(prompt)}")

    def run():
        for i in range(retries):
            try:
                res = client.responses.create(
                    model=model,
                    input=prompt,
                    temperature=temperature
                )
                return getattr(res, "output_text", "")
            except Exception as e:
                logger.warning(f"Retry {i+1}: {e}")
                time.sleep(1.5 * (i+1))

        raise RuntimeError("LLM failed after retries")

    return cached_call(key, run)

In [19]:
import json
import re


def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError(f"Invalid JSON output: {text[:200]}")

    try:
        return json.loads(match.group())
    except Exception as e:
        raise ValueError(f"JSON parse error: {e} | Raw: {text[:200]}")

### Parse Rubric

In [184]:
import re

def parse_rubric(rubric_text):

    # split sections
    case_section = re.search(
        r"Case study.*?Grading rubric.*?(?=Penalizations:|$)",
        rubric_text,
        re.DOTALL
    )

    penalty_section = re.search(
        r"Penalizations:(.*)",
        rubric_text,
        re.DOTALL
    )

    # -----------------------
    # TASK + SCENARIO
    # -----------------------
    task_block = case_section.group(0) if case_section else rubric_text

    instructions = []
    scenario_lines = []

    for line in task_block.split("\n"):
        l = line.strip()

        if not l:
            continue

        if "choose" in l.lower() or "apply" in l.lower():
            instructions.append(l)
        elif "scenario" in l.lower() or "smartcity" in l.lower():
            scenario_lines.append(l)

    # -----------------------
    # SCORING RULES
    # -----------------------
    per_action_rules = [
        {
            "name": "category_match",
            "weight": 0.5,
            "rule": "action corresponds to category"
        },
        {
            "name": "harm_description",
            "weight": 0.5,
            "rule": "clear harmful impact + specificity to SmartCity"
        }
    ]

    # -----------------------
    # PENALTIES
    # -----------------------
    penalties = []

    if penalty_section:
        for line in penalty_section.group(1).split("\n"):
            l = line.strip()
            if "-0.5" in l:
                penalties.append({
                    "name": "similar_actions",
                    "value": -0.5,
                    "rule": l
                })
            elif "-0.25" in l:
                penalties.append({
                    "name": "weak_impact",
                    "value": -0.25,
                    "rule": l
                })

    return {
        "task": {
            "name": "Case study",
            "total_points": 3,
            "instructions": instructions,
            "scenario": "\n".join(scenario_lines)
        },
        "scoring": {
            "per_action": per_action_rules,
            "penalties": penalties
        }
    }

In [185]:
# 1. STRUCTURE
rubric_structured = parse_rubric(rubric)

In [186]:
rubric_structured

{'task': {'name': 'Case study',
  'total_points': 3,
  'instructions': ['Case study (3 points): In this exercise, your task is to apply the Bad Actors Modeling strategy to the',
   'Choose 3 different categories among the five in the Bad Actors Modeling strategy (money,'],
  'scenario': 'SmartCity solution described in the scenario below:\nScenario: The Swiss company SmartCity wants to help local authorities to better manage city\nsolution proposed by SmartCity uses a range of connected devices to monitor waste-bins, traffic flow\nmonitor and control the system remotely from a centralized dashboard. SmartCity plans to roll out its\nand is specific to SmartCity (i.e. adapted to the scenario)'},
 'scoring': {'per_action': [{'name': 'category_match',
    'weight': 0.5,
    'rule': 'action corresponds to category'},
   {'name': 'harm_description',
    'weight': 0.5,
    'rule': 'clear harmful impact + specificity to SmartCity'}],
  'penalties': [{'name': 'similar_actions',
    'value': -0.

#### Analyze Rubric

In [23]:
MODEL_GRADING = "gpt-4.1"
MODEL_EXTRACTION = "gpt-4.1-mini"
MODEL_ANALYSIS = "gpt-4.1"

In [24]:


def analyze_rubric(rubric, teaching_material):
    prompt = f"""
You are an expert in exam design.

Your task is to evaluate how well the rubric aligns with the teaching material.

RUBRIC:
{rubric}

TEACHING MATERIAL:
{teaching_material}

IMPORTANT:
- Focus on alignment with teaching
- Identify missing concepts
- Identify unclear grading rules
- Do NOT invent new criteria

Evaluate it on:

1. Ambiguity → Is the rubric formulated with the objective, unambiguous criteria?
2. Applicability → Does the rubric cover the diversity of possible student responses?
3. Discrimination → Does the rubric clearly separate excellent from poor work?

Return STRICT JSON:

{{
  "ambiguity_issues": ["..."],
  "applicability_issues": ["..."],
  "discrimination_issues": ["..."],
  "alignment_issues": ["..."]
}}
"""
    output = call_llm(prompt, model= MODEL_ANALYSIS, temperature=0.4)
    return extract_json(output)

In [25]:
rubric_analysis = analyze_rubric(rubric, teaching)

In [26]:
rubric_analysis

{'ambiguity_issues': ["The rubric refers to 'the cheatsheet' for checking if an action corresponds to a category, but does not specify what constitutes sufficient alignment—this may be unclear to students or graders.",
  "The phrase 'actions must be different from each other' is not precisely defined; it is ambiguous how much overlap is allowed between actions.",
  "The penalty '-0.5 if two actions are too similar to each other' is subjective and does not define what 'too similar' means.",
  "The penalty '-0.25 if negative impact is not sufficiently described' does not specify what level of detail or specificity is required for sufficiency."],
 'applicability_issues': ['The rubric assumes students will choose three categories, but does not address how to grade if a student chooses fewer or more than three.',
  'There is no explicit guidance on how to handle partially correct answers, such as when an action fits a category but the negative impact is generic or only partially related to 

### Generate Synthetic Answers

In [27]:



def generate_synthetic_answers(teaching, rubric):
    """
    Generates calibration dataset:
    - good answers
    - bad answers
    - edge cases
    """

    prompt = f"""
You are generating synthetic student answers for rubric evaluation.

TEACHING MATERIAL:
{teaching}

RUBRIC:
{rubric}

Generate:

1. 4 HIGH-QUALITY answers (full understanding)
2. 4 LOW-QUALITY answers (misunderstanding / missing parts)
3. 3 EDGE CASE answers (partially correct but ambiguous)

Return STRICT JSON:

{{
  "good": ["..."],
  "bad": ["..."],
  "edge": ["..."]
}}
"""

    return extract_json(call_llm(prompt, MODEL_ANALYSIS, 0.4))

In [28]:
synthetic = generate_synthetic_answers(teaching, rubric)


In [29]:
synthetic

{'good': ['1. Money: A hacker could deploy ransomware on the SmartCity IoT network, locking city officials out of waste-bin monitoring and traffic control systems until a ransom is paid. This would disrupt essential city services, causing delays in waste collection and traffic congestion, negatively affecting residents and city operations.\n2. Politics: A nation-state actor could manipulate the traffic data to create artificial congestion during a major political event, making it difficult for attendees and emergency services to reach the area. This could undermine public trust in local authorities and disrupt civic activities.\n3. Self-Interest: An insider working for the city could alter weather sensor data to falsely indicate extreme heat, triggering unnecessary ventilation and cooling in public buildings. This would waste city resources and increase operational costs, impacting taxpayers.',
  '1. Ideas: Hacktivists could deface the public app that shares aggregated weather and poll

In [30]:
synthetic_good = synthetic["good"]
synthetic_bad = synthetic["bad"]

In [31]:
synthetic_edge = synthetic["edge"]

In [34]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 7.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [anthropic]/2 [anthropic]


In [46]:


import asyncio
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic
#from app.utils.json_utils import extract_json
#from app.config import OPENAI_API_KEY, ANTHROPIC_API_KEY

openai_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
claude_client = AsyncAnthropic(api_key=ANTHROPIC_API_KEY)


def build_prompt(rubric, answer):
    return f"""
Strict grader.

RUBRIC:
{rubric}

ANSWER:
{answer}

Return STRICT JSON:

{{
  "criteria_scores": [
    {{
      "criterion": "string",
      "max_score": number,
      "score": number,
      "reason": "string",
      "is_ambiguous": true
    }}
  ],
  "final_grade": number,
  "overall_reason": "string"
}}
"""


async def grade_answer_openai_async(rubric, answer):
    prompt = build_prompt(rubric, answer)

    res = await openai_client.responses.create(
        model="gpt-4.1",
        input=prompt,
        temperature=0.0
    )
    try:
        return extract_json(res.output_text)
    except Exception as e:
        return {
            "criteria_scores": [],
            "final_grade": 0,
            "overall_reason": "Parsing failed",
            "error": str(e),
            "raw_output": res.output_text
        }


async def grade_answer_claude_async(rubric, answer):
    prompt = build_prompt(rubric, answer)

    res = await claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        return extract_json(res.content[0].text)
    except Exception as e:
        return {
            "criteria_scores": [],
            "final_grade": 0,
            "overall_reason": "Parsing failed",
            "error": str(e),
            "raw_output": res.content[0].text
        }


async def grade_student(rubric, answer):
    o, c = await asyncio.gather(
        grade_answer_openai_async(rubric, answer),
        grade_answer_claude_async(rubric, answer)
    )

    return {"openai": o, "claude": c}

### Simulate Grading Async

In [47]:
import asyncio
#from app.grading.grader_async import grade_student


async def simulate_grading_async(rubric, students, runs=3):
    all_runs = []

    for _ in range(runs):
        tasks = [grade_student(rubric, s) for s in students]
        results = await asyncio.gather(*tasks)
        all_runs.append(results)

    return all_runs


async def simulate_all(rubric, students, synthetic):
    return await asyncio.gather(
        simulate_grading_async(rubric, students),
        simulate_grading_async(rubric, synthetic["good"]),
        simulate_grading_async(rubric, synthetic["bad"]),
        simulate_grading_async(rubric, synthetic["edge"])
    )


def build_rater_runs(grading_runs_multi):
    """
    Converts:
    runs x students x models

    → raters x students

    Each (run, model) = one rater for ICC
    """
    runs = []

    for run in grading_runs_multi:
        runs.append([r["openai"] for r in run])
        runs.append([r["claude"] for r in run])

    return runs

In [48]:
grading_runs_multi, good_runs_multi, bad_runs_multi, edge_runs_multi = await simulate_all(
    rubric, students, synthetic
)

/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py:984: RuntimeWarning: coroutine 'simulate_all' was never awaited
  exceptions.append(my_exceptions)


In [49]:
grading_runs_multi

[[{'openai': {'criteria_scores': [{'criterion': 'Politics',
      'max_score': 1,
      'score': 1,
      'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
      'is_ambiguous': False},
     {'criterion': 'Money',
      'max_score': 1,
      'score': 1,
      'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
      'is_ambiguous': False},
     {'criterion': 'Entertainment',
      'max_score': 1,
      'score': 1,
      'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly)

In [50]:
good_runs_multi

[[{'openai': {'criteria_scores': [{'criterion': 'Money - Action corresponds to category',
      'max_score': 0.5,
      'score': 0.5,
      'reason': "Deploying ransomware for financial gain is a classic 'money' bad actor motivation.",
      'is_ambiguous': False},
     {'criterion': 'Money - Negative impact described and scenario-specific',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'The answer clearly explains how ransomware would disrupt waste collection and traffic, harming residents and city operations.',
      'is_ambiguous': False},
     {'criterion': 'Politics - Action corresponds to category',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'Manipulating traffic data to influence a political event is a clear example of a politically motivated attack.',
      'is_ambiguous': False},
     {'criterion': 'Politics - Negative impact described and scenario-specific',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'The answer specifies disrup

In [51]:
bad_runs_multi

[[{'openai': {'criteria_scores': [{'criterion': 'Money',
      'max_score': 1,
      'score': 0.5,
      'reason': "The answer identifies hacking for money, which fits the 'money' category, but the harmful impact is vague ('bad for the city') and not specific to the SmartCity scenario.",
      'is_ambiguous': False},
     {'criterion': 'Politics',
      'max_score': 1,
      'score': 0.5,
      'reason': "The answer mentions using the system for political reasons, which fits the 'politics' category, but the negative impact is not described in a specific or detailed way ('could hurt people') and is not clearly linked to the SmartCity context.",
      'is_ambiguous': False},
     {'criterion': 'Entertainment',
      'max_score': 1,
      'score': 0.5,
      'reason': "The answer identifies playing a prank, which fits the 'entertainment' category, but the negative impact is only described as 'annoying' and lacks detail or specificity to the SmartCity scenario.",
      'is_ambiguous': Fals

In [52]:
edge_runs_multi

[[{'openai': {'criteria_scores': [{'criterion': 'Money',
      'max_score': 1,
      'score': 0.5,
      'reason': "The action (competitor accessing data) fits the 'money' category, but the method is vague and the negative impact on stakeholders (beyond SmartCity losing business) is not clearly described or specific to the scenario.",
      'is_ambiguous': True},
     {'criterion': 'Politics',
      'max_score': 1,
      'score': 0.5,
      'reason': "The action (influencing elections via the public app) fits the 'politics' category, but the method is not specified and the negative impact on stakeholders is not described in detail or adapted to the SmartCity context.",
      'is_ambiguous': True},
     {'criterion': 'Entertainment',
      'max_score': 1,
      'score': 0.5,
      'reason': "The action (messing with traffic lights for fun) fits the 'entertainment' category, but the impact on stakeholders is not fully explained or specific (e.g., how it could cause accidents or traffic j

In [53]:
openai_runs = [[r["openai"] for r in run] for run in grading_runs_multi]

In [133]:
all_raters = build_rater_runs(grading_runs_multi)

In [134]:
all_raters

[[{'criteria_scores': [{'criterion': 'Politics',
     'max_score': 1,
     'score': 1,
     'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
     'is_ambiguous': False},
    {'criterion': 'Money',
     'max_score': 1,
     'score': 1,
     'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
     'is_ambiguous': False},
    {'criterion': 'Entertainment',
     'max_score': 1,
     'score': 1,
     'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly) is clearly described an

 ### AGGREGATION

In [54]:
import numpy as np

def aggregate(grading_runs):
    result = []

    for i in range(len(grading_runs[0])):
        scores = [run[i]["final_grade"] for run in grading_runs]

        result.append({
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
            "min": float(np.min(scores)),
            "max": float(np.max(scores))
        })

    return result

def summarize_runs(aggregated):
    return [
        f"Student {i+1}: mean={a['mean']:.1f}, std={a['std']:.1f}"
        for i, a in enumerate(aggregated)
    ]

In [151]:
def aggregate_multi(grading_runs_multi):
    result = []

    num_students = len(grading_runs_multi[0])

    for i in range(num_students):
        scores = []

        for run in grading_runs_multi:
            scores.append(run[i]["openai"].get("final_grade", 0))
            scores.append(run[i]["claude"].get("final_grade", 0))

        result.append({
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
            "min": float(np.min(scores)),
            "max": float(np.max(scores))
        })

    return result

In [ ]:

aggregated_openai = aggregate(openai_runs)
aggregated_summary_openai = summarize_runs(aggregated_openai)

In [ ]:
aggregated_openai

[{'mean': 3.0, 'std': 0.0, 'min': 3.0, 'max': 3.0},
 {'mean': 1.5833333333333333,
  'std': 0.11785113019775792,
  'min': 1.5,
  'max': 1.75},
 {'mean': 2.8333333333333335,
  'std': 0.11785113019775792,
  'min': 2.75,
  'max': 3.0}]

In [ ]:
aggregated_summary_openai

['Student 1: mean=3.0, std=0.0',
 'Student 2: mean=1.6, std=0.1',
 'Student 3: mean=2.8, std=0.1']

In [152]:
aggregated = aggregate_multi(grading_runs_multi)
    

In [153]:
aggregated

[{'mean': 2.6750000000000003,
  'std': 0.3338537604001689,
  'min': 2.25,
  'max': 3.0},
 {'mean': 1.25, 'std': 0.5816642788871715, 'min': 0.0, 'max': 1.75},
 {'mean': 2.8333333333333335,
  'std': 0.18633899812498247,
  'min': 2.5,
  'max': 3.0}]

In [154]:
aggregated_summary = summarize_runs(aggregated)

In [155]:
aggregated_summary

['Student 1: mean=2.7, std=0.3',
 'Student 2: mean=1.2, std=0.6',
 'Student 3: mean=2.8, std=0.2']

## Test Evaluation

### Metrics

In [61]:
!pip install pandas scipy pingouin

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 34.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 31.7 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.5 MB/s  0:00:00eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 48.6 MB/s  0:00:00
Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 38.6 MB/s  0:00:00
Using cache

In [99]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import pingouin as pg


# ----------------------------
# 1. AMBIGUITY → ICC
# ----------------------------
def ambiguity_score_icc(grading_runs):
    """
    Measures inter-rater reliability using ICC.
    High ICC → low ambiguity.

    Uses:
    - ICC(A,1) → agreement between individual raters (preferred)
    - fallback: ICC(A,k)

    Returns score in [0,100]
    """

    data = []

    for r_idx, run in enumerate(grading_runs):
        for s_idx, g in enumerate(run):
            data.append({
                "student": s_idx,
                "rater": r_idx,
                "score": g.get("final_grade", 0)
            })

    df = pd.DataFrame(data)

    # Edge case: no variance → perfect agreement
    if df["score"].nunique() <= 1:
        return 100.0

    icc = pg.intraclass_corr(
        data=df,
        targets="student",
        raters="rater",
        ratings="score"
    )


    row = icc[icc["Type"].isin(["ICC(A,1)", "ICC(A,k)"])]

    if row.empty:
        return 0.0

    value = row["ICC"].iloc[0]

    # Clamp to valid ICC range
    value = max(-1.0, min(1.0, float(value)))

    # Map [-1,1] → [0,100]
    score = (value + 1) / 2 * 100
    return score


# ----------------------------
# 2. DISCRIMINATION → SPEARMAN
# ----------------------------
def discrimination_score_spearman(good, edge, bad):
    """
    Measures whether ranking is preserved:
    good > edge > bad using Spearman correlation.
    """

    scores = good + edge + bad

    # labels: good=2, edge=1, bad=0
    labels = (
        [2] * len(good) +
        [1] * len(edge) +
        [0] * len(bad)
    )

    if len(scores) < 2:
        return 0.0

    corr, _ = spearmanr(scores, labels)

    if np.isnan(corr):
        return 0.0

    # map [-1,1] → [0,100]
    return float((corr + 1) / 2 * 100)


# ----------------------------
# 3. APPLICABILITY → EDGE VARIANCE
# ----------------------------
def applicability_score(edge_scores, grading_runs):
    """
    Measures how rubric handles edge cases.
    Uses standard deviation:
    - too low → rigid
    - too high → unstable
    Ideal → moderate variance
    """

    if not edge_scores:
        return 0.0

    std = np.std(edge_scores)

    overall_std = np.std([
        g["final_grade"]
        for run in grading_runs
        for g in run
    ]) + 1e-6

    stability = 1 - (std / overall_std)

    return float(np.clip(stability, 0, 1) * 100)


# ----------------------------
# 4. CONSISTENCY → STD ACROSS RUNS
# ----------------------------
def consistency_score(grading_runs):
    """
    Measures within-model consistency (across runs).
    """

    stds = []

    for i in range(len(grading_runs[0])):
        vals = [r[i]["final_grade"] for r in grading_runs]
        stds.append(np.std(vals))

    mean_std = np.mean(stds)

    overall_std = np.std([
        g["final_grade"]
        for run in grading_runs
        for g in run
    ]) + 1e-6

    return float(np.clip(1 - mean_std / overall_std, 0, 1) * 100)


# ----------------------------
# 5. CROSS-MODEL CONSISTENCY
# ----------------------------
def cross_model_consistency(results):
    """
    Measures agreement between OpenAI and Claude.
    """

    diffs = []

    for r in results:
        o = r["openai"].get("final_grade", 0)
        c = r["claude"].get("final_grade", 0)
        diffs.append(abs(o - c))

    if not diffs:
        return 100.0

    max_diff = max(diffs) + 1e-6
    mean_diff = np.mean(diffs)

    return float(np.clip(1 - mean_diff / max_diff, 0, 1) * 100)

In [63]:
# 7. CONSISTENCY
consistency = consistency_score(openai_runs)

In [64]:
consistency

87.71411687524876

In [65]:
# 8. CROSS MODEL
flat_multi = [r for run in grading_runs_multi for r in run]
cross_model_score = cross_model_consistency(flat_multi)

In [66]:
cross_model_score

63.33335777776148

In [211]:
def extract_scores(runs, mode="all"):
    scores = []

    for run in runs:
        for r in run:
            if mode == "all":
                for model in r:
                    scores.append(r[model]["final_grade"])
            else:
                scores.append(r[mode]["final_grade"])

    return scores

In [210]:
good_runs_multi

[[{'openai': {'criteria_scores': [{'criterion': 'Money - Action corresponds to category',
      'max_score': 0.5,
      'score': 0.5,
      'reason': "Deploying ransomware for financial gain is a classic 'money' bad actor motivation.",
      'is_ambiguous': False},
     {'criterion': 'Money - Negative impact described and scenario-specific',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'The answer clearly explains how ransomware would disrupt waste collection and traffic, harming residents and city operations.',
      'is_ambiguous': False},
     {'criterion': 'Politics - Action corresponds to category',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'Manipulating traffic data to influence a political event is a clear example of a politically motivated attack.',
      'is_ambiguous': False},
     {'criterion': 'Politics - Negative impact described and scenario-specific',
      'max_score': 0.5,
      'score': 0.5,
      'reason': 'The answer specifies disrup

In [ ]:
 # 9. EXTRACT SCORES
good_scores_openai = extract_scores(good_runs_multi, "openai")
bad_scores_openai  = extract_scores(bad_runs_multi, "openai")
edge_scores_openai = extract_scores(edge_runs_multi, "openai")

In [ ]:
good_scores_openai

[3.0, 3, 3, 3, 3.0, 3, 3, 3, 3.0, 3, 3, 3]

In [ ]:
bad_scores_openai

[1.5, 1.5, 0.5, 1, 1.5, 1.5, 0, 1.5, 1.5, 1.5, 0, 1]

In [ ]:
edge_scores_openai

[1.5, 1.75, 2, 1.5, 1.75, 2, 1.5, 1.75, 2]

In [93]:
openai_runs

[[{'criteria_scores': [{'criterion': 'Politics',
     'max_score': 1,
     'score': 1,
     'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
     'is_ambiguous': False},
    {'criterion': 'Money',
     'max_score': 1,
     'score': 1,
     'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
     'is_ambiguous': False},
    {'criterion': 'Entertainment',
     'max_score': 1,
     'score': 1,
     'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly) is clearly described an

In [135]:
all_raters

[[{'criteria_scores': [{'criterion': 'Politics',
     'max_score': 1,
     'score': 1,
     'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
     'is_ambiguous': False},
    {'criterion': 'Money',
     'max_score': 1,
     'score': 1,
     'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
     'is_ambiguous': False},
    {'criterion': 'Entertainment',
     'max_score': 1,
     'score': 1,
     'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly) is clearly described an

## TEST SCORES

## Only Openai Runs

### Ambiguity (Inter-rater reliability)
### Intraclass Correlation (ICC)

In [94]:
data = []

for r_idx, run in enumerate(openai_runs):
    for s_idx, g in enumerate(run):
        data.append({
            "student": s_idx,
            "rater": r_idx,
            "score": g.get("final_grade", 0)
        })

df = pd.DataFrame(data)

In [95]:
df

,student,rater,score
0,0,0,3.00
1,1,0,1.50
2,2,0,2.75
3,0,1,3.00
4,1,1,1.75
5,2,1,2.75
6,0,2,3.00
7,1,2,1.50
8,2,2,3.00


In [96]:
df["score"].nunique()

4

In [97]:
icc = pg.intraclass_corr(
        data=df,
        targets="student",
        raters="rater",
        ratings="score"
    )

In [98]:
icc

,Type,ICC,F,df1,df2,pval,CI95
0,"ICC(1,1)",0.977186,129.5,2,6,0.000012,"[0.85, 1.0]"
1,"ICC(A,1)",0.977143,103.6,2,4,0.000359,"[0.83, 1.0]"
2,"ICC(C,1)",0.971591,103.6,2,4,0.000359,"[0.74, 1.0]"
3,"ICC(1,k)",0.992278,129.5,2,6,0.000012,"[0.94, 1.0]"
4,"ICC(A,k)",0.992263,103.6,2,4,0.000359,"[0.94, 1.0]"
5,"ICC(C,k)",0.990347,103.6,2,4,0.000359,"[0.9, 1.0]"


In [104]:
row = icc[icc["Type"].isin(["ICC(A,1)", "ICC(A,k)"])]

In [105]:
row

,Type,ICC,F,df1,df2,pval,CI95
1,"ICC(A,1)",0.977143,103.6,2,4,0.000359,"[0.83, 1.0]"
4,"ICC(A,k)",0.992263,103.6,2,4,0.000359,"[0.94, 1.0]"


In [106]:
value = row["ICC"].iloc[0]
value

np.float64(0.9771428571428573)

In [107]:
value = max(-1.0, min(1.0, float(value)))


In [108]:
value

0.9771428571428573

### DISCRIMINATION → SPEARMAN

In [110]:
scores = good_scores + edge_scores + bad_scores
scores

[3.0,
 3,
 3,
 3,
 3.0,
 3,
 3,
 3,
 3.0,
 3,
 3,
 3,
 1.5,
 1.75,
 2,
 1.5,
 1.75,
 2,
 1.5,
 1.75,
 2,
 1.5,
 1.5,
 0.5,
 1,
 1.5,
 1.5,
 0,
 1.5,
 1.5,
 1.5,
 0,
 1]

In [112]:
# labels: good=2, edge=1, bad=0
labels = (
        [2] * len(good_scores) +
        [1] * len(edge_scores) +
        [0] * len(bad_scores)
    )
labels

[2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [113]:
corr, _ = spearmanr(scores, labels)
corr

np.float64(0.938078085394056)

In [114]:
float((corr + 1) / 2 * 100)

96.9039042697028

### Applicability (EDGE VARIANCE)

In [115]:
std = np.std(edge_scores)
std

np.float64(0.2041241452319315)

In [117]:
overall_std = np.std([
        g["final_grade"]
        for run in openai_runs
        for g in run
    ]) + 1e-6

overall_std

np.float64(0.6394934685122966)

In [118]:
stability = 1 - (std / overall_std)
stability

np.float64(0.6808033931811667)

In [119]:
float(np.clip(stability, 0, 1) * 100)

68.08033931811667

### CONSISTENCY → STD ACROSS RUNS

In [120]:
stds = []

for i in range(len(openai_runs[0])):
    vals = [r[i]["final_grade"] for r in openai_runs]
    stds.append(np.std(vals))

stds

[np.float64(0.0),
 np.float64(0.11785113019775792),
 np.float64(0.11785113019775792)]

In [121]:
mean_std = np.mean(stds)
mean_std

np.float64(0.07856742013183861)

In [122]:
overall_std = np.std([
        g["final_grade"]
        for run in openai_runs
        for g in run
    ]) + 1e-6

overall_std

np.float64(0.6394934685122966)

In [123]:
stability = 1 - (mean_std / overall_std)
stability

np.float64(0.8771411687524876)

In [124]:
float(np.clip(stability, 0, 1) * 100)

87.71411687524876

## ALL RATERS ( OPENAI + CLAUDE)

In [212]:
 # 9. EXTRACT SCORES
good_scores = extract_scores(good_runs_multi)
bad_scores  = extract_scores(bad_runs_multi)
edge_scores = extract_scores(edge_runs_multi)

In [213]:
good_scores

[3.0,
 3.0,
 3,
 2.25,
 3,
 2.25,
 3,
 2.25,
 3.0,
 3.0,
 3,
 2.25,
 3,
 2.25,
 3,
 2.5,
 3.0,
 3.0,
 3,
 2.25,
 3,
 2.75,
 3,
 3.0]

In [214]:
bad_scores

[1.5,
 1.25,
 1.5,
 0,
 0.5,
 1.5,
 1,
 1.2,
 1.5,
 0.95,
 1.5,
 0.45,
 0,
 0.25,
 1.5,
 0,
 1.5,
 0.75,
 1.5,
 0,
 0,
 0.25,
 1,
 0.5]

In [216]:
edge_scores

[1.5,
 1.375,
 1.75,
 1.25,
 2,
 1.4,
 1.5,
 1.45,
 1.75,
 1.25,
 2,
 1.2,
 1.5,
 0,
 1.75,
 0.95,
 2,
 0]

### Ambiguity (Inter-rater reliability)
### Intraclass Correlation (ICC)

In [136]:
data = []

for r_idx, run in enumerate(all_raters):
    for s_idx, g in enumerate(run):
        data.append({
            "student": s_idx,
            "rater": r_idx,
            "score": g.get("final_grade", 0)
        })

df = pd.DataFrame(data)

In [137]:
df

,student,rater,score
0,0,0,3.00
1,1,0,1.50
2,2,0,2.75
3,0,1,2.25
4,1,1,0.00
5,2,1,3.00
6,0,2,3.00
7,1,2,1.75
8,2,2,2.75
9,0,3,2.30


In [138]:
df["score"].nunique()

10

In [139]:
icc = pg.intraclass_corr(
        data=df,
        targets="student",
        raters="rater",
        ratings="score"
    )

In [140]:
icc

,Type,ICC,F,df1,df2,pval,CI95
0,"ICC(1,1)",0.789786,23.542353,2,15,0.000024,"[0.4, 0.99]"
1,"ICC(A,1)",0.791083,28.610869,2,10,0.000073,"[0.4, 0.99]"
2,"ICC(C,1)",0.821486,28.610869,2,10,0.000073,"[0.41, 0.99]"
3,"ICC(1,k)",0.957523,23.542353,2,15,0.000024,"[0.8, 1.0]"
4,"ICC(A,k)",0.957841,28.610869,2,10,0.000073,"[0.8, 1.0]"
5,"ICC(C,k)",0.965048,28.610869,2,10,0.000073,"[0.81, 1.0]"


In [141]:
row = icc[icc["Type"].isin(["ICC(A,1)", "ICC(A,k)"])]

In [142]:
row

,Type,ICC,F,df1,df2,pval,CI95
1,"ICC(A,1)",0.791083,28.610869,2,10,0.000073,"[0.4, 0.99]"
4,"ICC(A,k)",0.957841,28.610869,2,10,0.000073,"[0.8, 1.0]"


In [143]:
value = row["ICC"].iloc[0]
value

np.float64(0.7910827644029228)

In [144]:
value = max(-1.0, min(1.0, float(value)))


In [145]:
value

0.7910827644029228

### DISCRIMINATION → SPEARMAN

In [217]:
scores = good_scores + edge_scores + bad_scores
scores

[3.0,
 3.0,
 3,
 2.25,
 3,
 2.25,
 3,
 2.25,
 3.0,
 3.0,
 3,
 2.25,
 3,
 2.25,
 3,
 2.5,
 3.0,
 3.0,
 3,
 2.25,
 3,
 2.75,
 3,
 3.0,
 1.5,
 1.375,
 1.75,
 1.25,
 2,
 1.4,
 1.5,
 1.45,
 1.75,
 1.25,
 2,
 1.2,
 1.5,
 0,
 1.75,
 0.95,
 2,
 0,
 1.5,
 1.25,
 1.5,
 0,
 0.5,
 1.5,
 1,
 1.2,
 1.5,
 0.95,
 1.5,
 0.45,
 0,
 0.25,
 1.5,
 0,
 1.5,
 0.75,
 1.5,
 0,
 0,
 0.25,
 1,
 0.5]

In [218]:
# labels: good=2, edge=1, bad=0
labels = (
        [2] * len(good_scores) +
        [1] * len(edge_scores) +
        [0] * len(bad_scores)
    )
labels

[2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [219]:
corr, _ = spearmanr(scores, labels)
corr

np.float64(0.8389032074818091)

In [221]:
float((corr + 1) / 2 * 100)

91.94516037409046

### Applicability (EDGE VARIANCE)

In [226]:
std = np.std(edge_scores)
std

np.float64(0.5612365783816842)

In [227]:
overall_std = np.std([
        g["final_grade"]
        for run in all_raters
        for g in run
    ]) + 1e-6

overall_std

np.float64(0.8175977870367639)

In [228]:
stability = 1 - (std / overall_std)
stability

np.float64(0.31355418607994867)

In [229]:
float(np.clip(stability, 0, 1) * 100)

31.355418607994867

### CONSISTENCY → STD ACROSS RUNS

In [169]:
stds = []

for i in range(len(openai_runs[0])):
    vals = [r[i]["final_grade"] for r in openai_runs]
    stds.append(np.std(vals))

stds

[np.float64(0.0),
 np.float64(0.11785113019775792),
 np.float64(0.11785113019775792)]

In [170]:
mean_std = np.mean(stds)
mean_std

np.float64(0.07856742013183861)

In [171]:
overall_std = np.std([
        g["final_grade"]
        for run in openai_runs
        for g in run
    ]) + 1e-6

overall_std

np.float64(0.6394934685122966)

In [172]:
stability = 1 - (mean_std / overall_std)
stability

np.float64(0.8771411687524876)

In [173]:
float(np.clip(stability, 0, 1) * 100)

87.71411687524876

###  CROSS-MODEL CONSISTENCY

In [125]:
flat_multi = [r for run in grading_runs_multi for r in run]
flat_multi

[{'openai': {'criteria_scores': [{'criterion': 'Politics',
     'max_score': 1,
     'score': 1,
     'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
     'is_ambiguous': False},
    {'criterion': 'Money',
     'max_score': 1,
     'score': 1,
     'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
     'is_ambiguous': False},
    {'criterion': 'Entertainment',
     'max_score': 1,
     'score': 1,
     'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly) is clearly de

In [126]:
diffs = []

for r in flat_multi:
    o = r["openai"].get("final_grade", 0)
    c = r["claude"].get("final_grade", 0)
    diffs.append(abs(o - c))

In [127]:
diffs

[0.75,
 1.5,
 0.25,
 0.7000000000000002,
 0.19999999999999996,
 0.25,
 0.5,
 0.30000000000000004,
 0.5]

In [128]:
max_diff = max(diffs) + 1e-6
mean_diff = np.mean(diffs)
max_diff, mean_diff

(1.500001, np.float64(0.55))

In [130]:
np.clip(1 - mean_diff / max_diff, 0, 1)

np.float64(0.6333335777776148)

In [131]:
float(np.clip(1 - mean_diff / max_diff, 0, 1) * 100)

63.33335777776148

In [175]:

def detect_rubric_failures(grading_runs, good_scores, edge_scores, bad_scores, cross_model_score):

    ambiguity = ambiguity_score_icc(grading_runs)
    discrimination = discrimination_score_spearman(good_scores, edge_scores, bad_scores)
    applicability = applicability_score(edge_scores, grading_runs)

    failures = []
    warnings = []
    insights = []

    # ----------------------------
    # 🚨 HARD FAILURES (rubric is broken)
    # ----------------------------

    if ambiguity < 75:
        failures.append({
            "type": "ambiguity",
            "message": "Low inter-rater agreement (ICC).",
            "score": ambiguity
        })

    if discrimination < 70:
        failures.append({
            "type": "discrimination",
            "message": "Rubric cannot distinguish good vs bad answers.",
            "score": discrimination
        })

    if applicability < 60:
        failures.append({
            "type": "applicability",
            "message": "Rubric fails on edge cases (too rigid or incomplete).",
            "score": applicability
        })

    if cross_model_score < 50:
        failures.append({
            "type": "cross_model",
            "message": "Severe disagreement between models → rubric highly ambiguous.",
            "score": cross_model_score
        })

    # ----------------------------
    # ⚠️ WARNINGS (subtle weaknesses)
    # ----------------------------

    # 1. Score saturation (important!)
    if ambiguity > 90 and np.std(good_scores) < 0.1:
        warnings.append({
            "type": "saturation",
            "message": "Top scores have near-zero variance → rubric too coarse or dataset too easy."
        })

    # 2. Cross-model disagreement (key signal)
    if 50 <= cross_model_score < 70:
        warnings.append({
            "type": "cross_model",
            "message": "Moderate disagreement between models → rubric wording ambigous."
        })

    # 3. Weak edge handling
    if 50 <= applicability < 70:
        warnings.append({
            "type": "edge_cases",
            "message": "Edge cases inconsistently graded → rubric lacks precision."
        })

    # 4. Overfitting to clear cases
    if discrimination > 90 and applicability < 70:
        warnings.append({
            "type": "overfit",
            "message": "Rubric works well for clear cases but struggles on borderline answers."
        })

    # 5. Hidden Ambiguity
    if ambiguity > 90 and cross_model_score < 70:
        warnings.append({
        "type": "hidden_ambiguity",
        "message": "High agreement within runs but disagreement across models → rubric wording unclear."
    })

     # ----------------------------
    # 🧠 INTERPRETABILITY INSIGHTS
    # ----------------------------

    if ambiguity > 90 and cross_model_score < 70:
        insights.append(
            "High agreement within model but disagreement across models → hidden ambiguity in wording."
        )

    if discrimination > 90 and np.std(good_scores) < 0.1:
        insights.append(
            "High discrimination but saturated top scores → rubric lacks granularity at high end."
        )

    if applicability < 70:
        insights.append(
            "Edge cases reveal missing rules or unclear boundaries in rubric."
        )


    return {
        "scores": {
            "ambiguity": round(ambiguity, 2),
            "discrimination": round(discrimination, 2),
            "applicability": round(applicability, 2),
            "cross_model": round(cross_model_score, 2)
        },
        "failures": failures, 
        "warnings" : warnings,
        "insights": insights
    }

In [230]:
# 10. EVALUATION
evaluation = detect_rubric_failures(
        openai_runs,
        good_scores,
        edge_scores,
        bad_scores,
        cross_model_score
    )

In [231]:
evaluation

{'scores': {'ambiguity': 98.86,
  'discrimination': 91.95,
  'applicability': 12.24,
  'cross_model': 63.33},
 'failures': [{'type': 'applicability',
   'message': 'Rubric fails on edge cases (too rigid or incomplete).',
   'score': 12.237324379976467}],
 'warnings': [{'type': 'cross_model',
   'message': 'Moderate disagreement between models → rubric wording ambigous.'},
  {'type': 'overfit',
   'message': 'Rubric works well for clear cases but struggles on borderline answers.'},
  {'type': 'hidden_ambiguity',
   'message': 'High agreement within runs but disagreement across models → rubric wording unclear.'}],
 'insights': ['High agreement within model but disagreement across models → hidden ambiguity in wording.',
  'Edge cases reveal missing rules or unclear boundaries in rubric.']}

In [177]:
evaluation

{'scores': {'ambiguity': 98.86,
  'discrimination': 96.9,
  'applicability': 68.08,
  'cross_model': 63.33},
 'failures': [],
 'warnings': [{'type': 'saturation',
   'message': 'Top scores have near-zero variance → rubric too coarse or dataset too easy.'},
  {'type': 'cross_model',
   'message': 'Moderate disagreement between models → rubric wording ambigous.'},
  {'type': 'edge_cases',
   'message': 'Edge cases inconsistently graded → rubric lacks precision.'},
  {'type': 'overfit',
   'message': 'Rubric works well for clear cases but struggles on borderline answers.'},
  {'type': 'hidden_ambiguity',
   'message': 'High agreement within runs but disagreement across models → rubric wording unclear.'}],
 'insights': ['High agreement within model but disagreement across models → hidden ambiguity in wording.',
  'High discrimination but saturated top scores → rubric lacks granularity at high end.',
  'Edge cases reveal missing rules or unclear boundaries in rubric.']}

## Build Prompt for Rubric Improvement

In [187]:
def build_prompt(rubric, rubric_structured, teaching, students, aggregated_summary, consistency, cross_model_score, evaluation, rubric_analysis):

    student_block = "\n\n".join(
        [f"Student {i+1}:\n{s}" for i, s in enumerate(students)]
    )

    return f"""
You are an expert exam designer.

TASK:
Improve the grading rubric using MULTIPLE sources of evidence based on the following goals.

--- INPUTS ---

RUBRIC:
{rubric}

STRUCTURED RUBRIC
{rubric_structured}

RUBRIC ANALYSIS:
{rubric_analysis}

TEACHING MATERIAL:
{teaching}

STUDENT ANSWERS:
{student_block}

AGGREGATED SUMMARY
{aggregated_summary}

CONSISTENCY SCORE
{consistency}

CROSS-MODEL-CONSISTENCY
{cross_model_score}

EVALUATION:
{evaluation}

FAILURES DETECTED:
{evaluation["failures"]}

WARNINGS:
{evaluation["warnings"]}

INSIGHTS:
{evaluation["insights"]}


--- TASK ---

Improve the rubric by:
- Reducing ambiguity (make criteria precise)
- Increasing applicability (cover more answer types)
- Improving discrimination (differentiate quality levels)

Goals:
1. Ambiguity → All the graders reach the same interpretation independently.
2. Applicability → No valid answer type is left unaddressed by the rubric.
3. Discrimination → High-quality answers score significantly better than weak ones.

INTERPRETATION GUIDELINES:

- If ambiguity is high but cross-model consistency is low:
  → criteria wording is ambiguous, not precise enough

- If applicability is low:
  → rubric is missing rules for borderline answers

- If discrimination is high but top scores have low variance:
  → rubric lacks granularity at high performance levels

- If edge cases show inconsistent scores:
  → scoring boundaries are not clearly defined

Use these signals to:
- rewrite vague criteria into measurable rules
- define clear scoring boundaries
- add examples for borderline cases

Use these signals to identify concrete weaknesses in rubric design.

OUTPUT STRICT JSON:
{{
  "improved_rubric": "...",
  "explanation": {{
    "ambiguity": "...",
    "applicability": "...",
    "discrimination": "..."
  }}
}}
"""

In [232]:
# 11. PROMPT
prompt = build_prompt(
        rubric,
        rubric_structured,
        teaching,
        students,
        aggregated_summary,   
        consistency,
        cross_model_score,
        evaluation,
        rubric_analysis
    )

In [233]:


def improve_rubric(prompt):
    out = call_llm(prompt, MODEL_GRADING, temperature= 0.4)
    return extract_json(out)

In [234]:
improved = improve_rubric(prompt)

In [235]:
improved

{'improved_rubric': {'task': {'name': 'Case study',
   'total_points': 3,
   'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.',
    'Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).',
    'For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.'],
   'scenario': 'SmartCity solution as described above.'},
  'scoring': {'per_action': [{'name': 'category_match',
     'weight': 0.5,
     'rule': "The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet)

In [236]:
print(improved["improved_rubric"])

{'task': {'name': 'Case study', 'total_points': 3, 'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.', 'Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).', 'For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.'], 'scenario': 'SmartCity solution as described above.'}, 'scoring': {'per_action': [{'name': 'category_match', 'weight': 0.5, 'rule': "The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet)."}, {'name': 'harm_description', 'weight': 0.5, 

In [237]:
improved["explanation"]

{'ambiguity': 'Ambiguity is reduced by providing explicit definitions, requiring clear links to both category motivation and scenario features, and by including quality levels with examples. The rubric now specifies what constitutes a good match to a category and a sufficiently detailed harm description. Borderline guidance and explicit reference to the cheatsheet/teaching material further reduce grader subjectivity.',
 'applicability': 'Applicability is increased by adding rules for partial answers, duplicate/missing categories, and overlapping motivations. The rubric now covers cases where students provide fewer than three actions, use the same category twice, or choose ambiguous actions. Coverage rules ensure all plausible answer types are addressed.',
 'discrimination': 'Discrimination is improved by introducing a 5-level quality scale (Excellent, Good, Adequate, Weak, Incorrect) for each action, with clear criteria and examples. This allows graders to differentiate between minimal

## Test Human Review

In [194]:
import json

def human_review(result):

    print("\n=== IMPROVED RUBRIC ===\n")
    print(result["improved_rubric"])

    print("\n=== EXPLANATION ===\n")
    print(json.dumps(result["explanation"], indent=2))

    choice = input("\nApprove? (y/n): ")

    if choice.lower() == "y":
        result["human_verified"] = True
        return result

    print("\nEdit rubric:")
    result["improved_rubric"] = input("> ")

    print("\nEdit explanation JSON:")
    try:
        result["explanation"] = json.loads(input("> "))
    except:
        pass

    result["human_verified"] = True
    return result

In [204]:
human_review(improved) 


=== IMPROVED RUBRIC ===

{'task': {'name': 'Case study', 'total_points': 3, 'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.', 'Select 3 different categories from: Money, Politics, Entertainment, Ideas, Self-Interest.', 'For each selected category: (a) Identify a plausible harmful action a bad actor could take, and (b) clearly explain how this action negatively impacts specific stakeholders in the SmartCity scenario (1-2 sentences per action).', 'Each action must be distinct in both method and impact.'], 'scenario': 'SmartCity solution as described above.'}, 'scoring': {'per_action': [{'name': 'category_match', 'weight': 0.5, 'rule': "The action clearly fits the chosen motivation category, as defined below. Use the category definitions and examples provided. If an action plausibly fits more than one category, the explanation must justify the chosen category based on the actor's primary motivation."}, {'name': 'harm_description', 'weight': 0.5, 'rule'

{'improved_rubric': {'task': {'name': 'Case study',
   'total_points': 3,
   'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.',
    'Select 3 different categories from: Money, Politics, Entertainment, Ideas, Self-Interest.',
    'For each selected category: (a) Identify a plausible harmful action a bad actor could take, and (b) clearly explain how this action negatively impacts specific stakeholders in the SmartCity scenario (1-2 sentences per action).',
    'Each action must be distinct in both method and impact.'],
   'scenario': 'SmartCity solution as described above.'},
  'scoring': {'per_action': [{'name': 'category_match',
     'weight': 0.5,
     'rule': "The action clearly fits the chosen motivation category, as defined below. Use the category definitions and examples provided. If an action plausibly fits more than one category, the explanation must justify the chosen category based on the actor's primary motivation."},
    {'name': 'harm_desc

In [205]:
human_reviewed_rubric = human_review(improved)


=== IMPROVED RUBRIC ===

{'task': {'name': 'Case study', 'total_points': 3, 'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.', 'Select 3 different categories from: Money, Politics, Entertainment, Ideas, Self-Interest.', 'For each selected category: (a) Identify a plausible harmful action a bad actor could take, and (b) clearly explain how this action negatively impacts specific stakeholders in the SmartCity scenario (1-2 sentences per action).', 'Each action must be distinct in both method and impact.'], 'scenario': 'SmartCity solution as described above.'}, 'scoring': {'per_action': [{'name': 'category_match', 'weight': 0.5, 'rule': "The action clearly fits the chosen motivation category, as defined below. Use the category definitions and examples provided. If an action plausibly fits more than one category, the explanation must justify the chosen category based on the actor's primary motivation."}, {'name': 'harm_description', 'weight': 0.5, 'rule'

## Test Improved Rubric

#### Evaluate new rubric

In [278]:
improved

{'improved_rubric': {'task': {'name': 'Case study',
   'total_points': 3,
   'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.',
    'Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).',
    'For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.'],
   'scenario': 'SmartCity solution as described above.'},
  'scoring': {'per_action': [{'name': 'category_match',
     'weight': 0.5,
     'rule': "The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet)

In [287]:
improved_rubric = improved["improved_rubric"]

In [288]:
print(improved_rubric)

{'task': {'name': 'Case study', 'total_points': 3, 'instructions': ['Apply the Bad Actors Modeling strategy to the SmartCity scenario.', 'Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).', 'For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.'], 'scenario': 'SmartCity solution as described above.'}, 'scoring': {'per_action': [{'name': 'category_match', 'weight': 0.5, 'rule': "The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet)."}, {'name': 'harm_description', 'weight': 0.5, 

In [267]:
def rubric_to_text(rubric):
    if isinstance(rubric, str):
        return rubric

    task = rubric.get("task", {})
    scoring = rubric.get("scoring", {})

    lines = []

    # Task
    lines.append(f"{task.get('name', 'Task')} ({task.get('total_points', 0)} points)")
    for instr in task.get("instructions", []):
        lines.append(f"- {instr}")

    # Scenario
    if task.get("scenario"):
        lines.append("\nScenario:")
        lines.append(task["scenario"])

    # Scoring
    lines.append("\nGrading rubric:")
    for rule in scoring.get("per_action", []):
        lines.append(f"- {rule.get('weight', '')}: {rule.get('rule', '')}")

    # Penalties
    if scoring.get("penalties"):
        lines.append("\nPenalties:")
        for p in scoring["penalties"]:
            lines.append(f"- {p.get('value', '')}: {p.get('rule', '')}")

    return "\n".join(lines)

In [289]:
improved_rubric_text = rubric_to_text(improved_rubric)

In [292]:
print(improved_rubric_text)

Case study (3 points)
- Apply the Bad Actors Modeling strategy to the SmartCity scenario.
- Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).
- For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.

Scenario:
SmartCity solution as described above.

Grading rubric:
- 0.5: The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet).
- 0.5: The negative impact is specifically described, naming at least one affected stakeholder group (e.g., city residents, local authorities, company, serv

In [283]:
improved_rubric = improved_rubric_text

In [285]:
print(improved_rubric_text)

Case study (3 points)
- Apply the Bad Actors Modeling strategy to the SmartCity scenario.
- Choose 3 different categories among the five in the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest).
- For each chosen category: (a) Identify one plausible harmful action a bad actor could take (1 sentence), and (b) Explain specifically how this action could negatively impact stakeholders in the SmartCity scenario (1-2 sentences). Actions must be distinct in both actor motivation and system impact.

Scenario:
SmartCity solution as described above.

Grading rubric:
- 0.5: The action is clearly and plausibly motivated by the chosen category, as defined in the Bad Actors Modeling teaching material. The motivation must be explicit and align with the category's definition and examples (see provided cheatsheet).
- 0.5: The negative impact is specifically described, naming at least one affected stakeholder group (e.g., city residents, local authorities, company, serv

In [259]:
def build_prompt(rubric, answer):
    return f"""
Strict grader.

RUBRIC:
{rubric}

ANSWER:
{answer}

Return STRICT JSON:

{{
  "criteria_scores": [
    {{
      "criterion": "string",
      "max_score": number,
      "score": number,
      "reason": "string",
      "is_ambiguous": true
    }}
  ],
  "final_grade": number,
  "overall_reason": "string"
}}
"""

In [260]:
# 1. STRUCTURE
rubric_structured = improved_rubric

# 2. ANALYZE
rubric_analysis = analyze_rubric(improved_rubric, teaching)

# 3. SYNTHETIC
synthetic_improved = generate_synthetic_answers(teaching, improved_rubric)

# 4. RUN ALL SIMULATIONS (ASYNC)
grading_runs_improved, good_runs_improved, bad_runs_improved, edge_runs_improved = await simulate_all(improved_rubric, students, synthetic_improved)


# 5. 
openai_runs_improved = [[r["openai"] for r in run] for run in grading_runs_improved]
all_raters_improved = build_rater_runs(grading_runs_improved)

# 6. AGGREGATION
#aggregated = aggregate(openai_runs)
aggregated_improved = aggregate_multi(grading_runs_improved)
aggregated_summary_improved = summarize_runs(aggregated_improved)

# 7. CONSISTENCY
consistency = consistency_score(openai_runs_improved)

# 8. CROSS MODEL
flat_multi_improved = [r for run in grading_runs_improved for r in run]
cross_model_score_improved = cross_model_consistency(flat_multi_improved)

# 9. EXTRACT SCORES
good_scores_improved = extract_scores(good_runs_improved)
bad_scores_improved  = extract_scores(bad_runs_improved)
edge_scores_improved = extract_scores(edge_runs_improved)

# 10. EVALUATION
evaluation_improved = detect_rubric_failures(
        all_raters_improved, #openai_runs,
        good_scores_improved,
        edge_scores_improved,
        bad_scores_improved, 
        cross_model_score_improved
    )


In [261]:
evaluation_improved

{'scores': {'ambiguity': 54.15,
  'discrimination': 84.2,
  'applicability': 79.74,
  'cross_model': 22.22},
 'failures': [{'type': 'ambiguity',
   'message': 'Low inter-rater agreement (ICC).',
   'score': 54.152484683458134},
  {'type': 'cross_model',
   'message': 'Severe disagreement between models → rubric highly ambiguous.',
   'score': 22.2222481481395}],
 'warnings': [],
 'insights': []}

In [208]:
baseline_runs = grading_runs_multi

In [209]:
baseline_runs

[[{'openai': {'criteria_scores': [{'criterion': 'Politics',
      'max_score': 1,
      'score': 1,
      'reason': "The action (intelligence agencies using the system to gather information) fits the 'politics' category and the negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to the SmartCity scenario.",
      'is_ambiguous': False},
     {'criterion': 'Money',
      'max_score': 1,
      'score': 1,
      'reason': "The action (robbers stealing valuable installed materials) fits the 'money' category and the negative impact (financial cost to reinstall materials) is clearly described and specific to the scenario.",
      'is_ambiguous': False},
     {'criterion': 'Entertainment',
      'max_score': 1,
      'score': 1,
      'reason': "The action (trolling waste collection services by manipulating sensors) fits the 'entertainment' category and the negative impact (waste collection services being misdirected repeatedly)

In [ ]:
improved_runs = simulate_grading_async(improved_rubric,students )

In [90]:
improved_runs

[[{'criteria_scores': [{'criterion': 'Politics - Category Alignment',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The action (intelligence agents using SmartCity to gain information) is clearly motivated by political interests, as defined in the rubric.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Action Specificity & Plausibility',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The action is specific (intelligence agents using SmartCity installations in foreign countries for espionage) and plausible in the scenario.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Scenario-Specific Negative Impact',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to SmartCity stakeholders.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Depth & Insight',
     'max_score': 0.25,
     'score': 0.1,
     

#### Compute metrics for both

In [91]:
baseline_eval = evaluation

In [92]:
baseline_eval

{'scores': {'ambiguity': 90.19,
  'discrimination': 18.35,
  'applicability': 44.44},
 'failures': ['Discrimination: Rubric does not separate strong and weak answers well.']}

In [93]:
improved_eval = detect_rubric_failures(improved_runs)

In [94]:
improved_eval

{'scores': {'ambiguity': 82.87,
  'discrimination': 23.15,
  'applicability': 55.56},
 'failures': ['Discrimination: Rubric does not separate strong and weak answers well.']}

In [96]:
def compare(baseline, improved):
    return {
        "ambiguity_improvement": improved["scores"]["ambiguity"] - baseline["scores"]["ambiguity"],
        "discrimination_improvement": improved["scores"]["discrimination"] - baseline["scores"]["discrimination"],
        "applicability_improvement": improved["scores"]["applicability"] - baseline["scores"]["applicability"]
    }

In [97]:
compare(baseline_eval, improved_eval)

{'ambiguity_improvement': -7.319999999999993,
 'discrimination_improvement': 4.799999999999997,
 'applicability_improvement': 11.120000000000005}